In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc


In [ ]:
H5AD = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/05b_trajectory_revisit/adata_CD8_subset_clusters_0_1_2_4.h5ad"
ad = sc.read_h5ad(H5AD)
print(ad)

In [ ]:
print(ad.obs['leiden_T_0.6'].value_counts())

mapping = {"4": "1", "0": "2", "1": "3", "2": "4"}
ad.obs["leiden_T_0.6_relabel"] = ad.obs["leiden_T_0.6"].astype(str).map(mapping)

# 如果你想覆盖原列：
# ad.obs["leiden_T_0.6"] = ad.obs["leiden_T_0.6"].astype(str).map(mapping)

# 可选：转回整数类型
ad.obs["leiden_T_0.6_relabel"] = ad.obs["leiden_T_0.6_relabel"].astype(int)

print(ad.obs["leiden_T_0.6_relabel"].value_counts().sort_index())




In [ ]:
# =========================
# DE settings
# =========================
METHOD = "wilcoxon"
N_GENES = None   # None = keep all genes; or set e.g. 5000
USE_LAYER = "log1p_norm"  # 若 adata.layers 里没有该层，会自动回退到 adata.X

# =========================
# Column hints (optional)
# =========================
# 自动检测失败时，手动指定：
CLUSTER_COL = 'leiden_T_0.6_relabel'   # e.g. "leiden" or "cluster" or "leiden_cd8"
RESPONSE_COL = 'Respond'  # e.g. "Responder" or "response" or "RNR" or "is_responder"


In [ ]:
ad.X = np.log1p(ad.layers["norm_expr"])

In [ ]:
ad.obs["RNR"] = ad.obs[RESPONSE_COL].astype(str).map({
    "Responder": "R",
    "Non-Responder": "NR"
})

# 只保留 R/NR（安全）
ad = ad[ad.obs["RNR"].isin(["R", "NR"])].copy()

# 确保 cluster 是字符串
ad.obs["cluster"] = ad.obs[CLUSTER_COL].astype(str)

In [ ]:
ad.obs["cluster"].value_counts()

In [ ]:
DE_DIR = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/07_perturb_seq_T_cells"
os.makedirs(DE_DIR, exist_ok=True)

DE_2TO3   = os.path.join(DE_DIR, "DE_cluster2_vs_cluster3.fullV2.csv")     # 4 -> 1 uses 1 vs 4
DE_4TO3   = os.path.join(DE_DIR, "DE_cluster4_vs_cluster3.fullV2.csv")     # 4 -> 2 uses 2 vs 4
DE_2TO4= os.path.join(DE_DIR, "DE_cluster2_vs_cluster4.fullV2.csv")         # within cluster4: R vs NR
DE_C3_RVSNR= os.path.join(DE_DIR, "DE_cluster3_R_vs_NR.fullV2.csv")         # within cluster1: R vs NR

In [ ]:
sc.tl.rank_genes_groups(
    ad,
    groupby="cluster",
    groups=["3"],
    reference="2",
    method="wilcoxon",
    use_raw=False,
    n_genes=ad.n_vars,
)
df = sc.get.rank_genes_groups_df(ad, group="3")   # ✅ 必须是 "3"
df.to_csv(DE_2TO3, index=False)
print("saved:", DE_2TO3, "rows=", df.shape[0])

# 4 -> 3 (we implement as DE: 3 vs 4)
sc.tl.rank_genes_groups(
    ad,
    groupby="cluster",
    groups=["3"],
    reference="4",
    method="wilcoxon",
    use_raw=False,
    n_genes=ad.n_vars,
)
df = sc.get.rank_genes_groups_df(ad, group="3")   # ✅ 仍然是 "3"
df.to_csv(DE_4TO3, index=False)
print("saved:", DE_4TO3, "rows=", df.shape[0])

sc.tl.rank_genes_groups(
    ad,
    groupby="cluster",
    groups=["4"],          # destination state
    reference="2",         # source state
    method="wilcoxon",
    use_raw=False,
    n_genes=ad.n_vars,
)
df = sc.get.rank_genes_groups_df(ad, group="4")  # ✅ must match groups=["4"]
df.to_csv(DE_2TO4, index=False)
print("saved:", DE_2TO4, "rows=", df.shape[0])

ad_c3 = ad[ad.obs["cluster"].astype(str) == "3"].copy()

# IMPORTANT: make sure ad_c3.obs has a binary label column for response
# e.g. ad_c3.obs["response"] in {"R","NR"}
sc.tl.rank_genes_groups(
    ad_c3,
    groupby="RNR",    # <-- change to your actual column name
    groups=["R"],          # destination phenotype
    reference="NR",        # source phenotype
    method="wilcoxon",
    use_raw=False,
    n_genes=ad_c3.n_vars,
)
df = sc.get.rank_genes_groups_df(ad_c3, group="R")  # ✅ must match groups=["R"]
df.to_csv(DE_C3_RVSNR, index=False)
print("saved:", DE_C3_RVSNR, "rows=", df.shape[0])